In [5]:
# ======================================================
# CIND820 - Keystone Project
# Data Analysis and Visualization: Fire Peril Loss Cost (Liberty Mutual Insurance)
# Author: Debra Allen 
# ======================================================

# Call Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [13]:
# =======================================================
# Data Preparation for Modeling: Fire Peril Loss Cost
# =======================================================

# Import necessary libraries for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [6]:
# ======================================================
# Load the dataset and display basic information
# ======================================================

# Use a smaller sample for modeling to avoid memory issues
def find_repo_root(start: Path | None = None) -> Path:
    """Find the root of the git repository."""
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    
    markers = [".git", "pyproject.toml", "requirements.txt"]
    for p in [start, *start.parents]:
        if any((p / m).exists() for m in markers):
            return p
    raise FileNotFoundError("Could not find repo root. Run notebook from inside the repo.")

REPO_ROOT = find_repo_root()
SAMPLE_PATH = REPO_ROOT /"liberty_train_subset.csv"
print("Repo Root:", REPO_ROOT)
print("Data Path:", SAMPLE_PATH)

# Load a smaller sample of the dataset for modeling
model_data = pd.read_csv(SAMPLE_PATH, low_memory=False)

# Display basic information about the dataset
model_data.info()

Repo Root: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820
Data Path: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820\liberty_train_subset.csv
<class 'pandas.DataFrame'>
RangeIndex: 113015 entries, 0 to 113014
Columns: 302 entries, id to weatherVar236
dtypes: float64(291), int64(1), str(10)
memory usage: 260.4 MB


In [ ]:
# ======================================================
# Prepare the dataset for modeling
# ======================================================

# Convert categorical variables to string type (if any)
for col in model_data.select_dtypes(include=["object"]).columns:
    model_data[col] = model_data[col].astype(str)

# Define the target variable and predictor variables
target = "target"
X = model_data.drop(columns=[target])
y = model_data[target]

C:\Users\uni_f\AppData\Local\Temp\ipykernel_35768\3056448203.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in model_data.select_dtypes(include=["object"]).columns:


In [8]:
# ======================================================
# Identify Variable Types
# ======================================================

# Identify categorical and numerical variables
categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()
numerical_vars = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Display variable types
print("Categorical Variables:", categorical_vars)
print("Numerical Variables:", numerical_vars)

# Display number of numerical and categorical variables
print("\nNumber of Categorical Variables:", len(categorical_vars))
print("Number of Numerical Variables:", len(numerical_vars))

Categorical Variables: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'dummy']
Numerical Variables: ['id', 'var10', 'var11', 'var12', 'var13', 'var14', 'var15', 'var16', 'var17', 'crimeVar1', 'crimeVar2', 'crimeVar3', 'crimeVar4', 'crimeVar5', 'crimeVar6', 'crimeVar7', 'crimeVar8', 'crimeVar9', 'geodemVar1', 'geodemVar2', 'geodemVar3', 'geodemVar4', 'geodemVar5', 'geodemVar6', 'geodemVar7', 'geodemVar8', 'geodemVar9', 'geodemVar10', 'geodemVar11', 'geodemVar12', 'geodemVar13', 'geodemVar14', 'geodemVar15', 'geodemVar16', 'geodemVar17', 'geodemVar18', 'geodemVar19', 'geodemVar20', 'geodemVar21', 'geodemVar22', 'geodemVar23', 'geodemVar24', 'geodemVar25', 'geodemVar26', 'geodemVar27', 'geodemVar28', 'geodemVar29', 'geodemVar30', 'geodemVar31', 'geodemVar32', 'geodemVar33', 'geodemVar34', 'geodemVar35', 'geodemVar36', 'geodemVar37', 'weatherVar1', 'weatherVar2', 'weatherVar3', 'weatherVar4', 'weatherVar5', 'weatherVar6', 'weatherVar7', 'weatherVar8', 'weatherVar9

C:\Users\uni_f\AppData\Local\Temp\ipykernel_35768\2312351895.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()


In [11]:
# ======================================================
# Preprocess the dataset
# ======================================================

# Numerical pipeline: impute missing values with median and scale features
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute missing values with most frequent and one-hot encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine numerical and categorical pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_vars),
    ('cat', categorical_pipeline, categorical_vars)
],
remainder='drop',
n_jobs=-1
)

In [12]:
# =======================================================
# Train-Test Split
# =======================================================

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])


Training observations: 90412
Testing observations: 22603


In [15]:
# ======================================================
# Tweedie GLM Model
# ======================================================

tweedie_glm = TweedieRegressor(
    power=1.5,  # Tweedie distribution with power between 1 and 2 for insurance claims
    alpha=0.1,  # Regularization strength
    link='log',
    max_iter=1000
)